# 🎯 [OPS02_M02] 이종 센서 시각 동기화(Time-sync) 및 공간 좌표계(ENU/ECEF) 변환 전처리 구현
### Google Colab 실습 환경 (작전·전투 수행 / Level 3)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/OPS02/ops02_m02_solution.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 작전·전투 수행 (Level 3) |
| **미션 ID** | OPS02_M02 |
| **미션 주제** | 이종 센서 시각 동기화(Time-sync) 및 공간 좌표계(ENU/ECEF) 변환 전처리 구현 |
| **소요 시간** | 약 2시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
미국-이란 간 무력 충돌 사례에서는 드론·미사일이 여러 방향에서 동시에 들어오면서 레이더, 광학(EO/IR), 음향 센서가 같은 비행체를 각자 경보해 상황판에 중복 항적이 쏟아지고, 이 때문에 위협 수를 잘못 세거나 대응이 늦어지는 문제가 주목받았다. 본 프로젝트에서는 가상 방어 지역에 배치한 가상 센서 4기와 가상 비행체 궤적으로 교육용 경보 데이터를 코드로 만들어, 연동 규격과 통합 스키마 설계, 시각 동기화와 ENU 좌표 변환 전처리, 최단거리 매칭 Baseline, 확장 칼만 필터(EKF) 항적 추적·통합, 융합 신뢰도 지수, 전술 상황판(COP), 조작관 HITL 검토, 통신 단절·재밍 스트레스 시험, AAR 재보정, 최종 통합까지 다중 센서 중복 항적 통합 시뮬레이터를 단계별로 구현한다. 최종 목표는 비행체 하나를 하나의 항적으로 정확히 보여 주어 지휘관의 인지 부하를 줄이고, 항적 통합·분리 같은 판단은 조작관이 근거와 함께 확정하도록 설계하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

---

## 📂 2. 산출물 디렉토리 체계 및 실습 가이드

본 과제의 미션별 산출물(`deliverables`)은 단순 스크립트를 넘어 실무 엔터프라이즈 및 국방 체계에 즉시 배포할 수 있는 **표준 패키지 디렉토리 구조**를 따릅니다:

| 디렉토리 | 주요 산출물(Deliverables) | 역할 및 설명 |
| :--- | :--- | :--- |
| **`data/`** | `data/schema/ops02_input_schema.json` | 입출력 데이터 표준 스키마 및 원천/정제 데이터셋 |
| **`src/`** | `src/preprocess/`, `src/models/`, `src/ui/`, `src/hitl/` | 전처리, 모델, 대시보드, HITL 승인 등 핵심 파이썬 소스 코드 모듈 |
| **`docs/`** | `docs/ops02_data_spec.md`, `docs/ops02_hitl_guide.md`, `docs/ops02_final_report.md` | 데이터 규격서, HITL 가이드, AAR 분석 보고서 등 실무/지휘관 문서 |
| **`configs/`**| `configs/ops02_tuned_params.yaml` | AAR 피드백 기반 재조정 임계치 및 환경설정 파일 |
| **`tests/`** | `tests/ops02_stress_test.py` | 500% 부하 시험 및 pytest 단위/회귀 테스트 스위트 |

> **💡 [Google Colab / 주피터 노트북 실습 팁]**
> - **방법 A. 자동 파일 저장 (추천):** 코드 셀 첫 줄에 `%%writefile src/models/ops02_baseline_model.py`와 같이 매직 명령어를 추가하여 실행하면, 노트북에서 작업한 코드가 해당 폴더의 파일로 자동 추출·저장됩니다.
> - **방법 B. 논리적 모듈 대응:** 별도 파일 추출 없이 노트북 셀 단위로 순차 실행하더라도 각 미션이 해당 모듈의 역할을 1:1로 수행하며, **미션 10(시스템 통합)**에서 전체 파이프라인이 유기적으로 완성되도록 설계되어 있습니다.

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 02. 이종 센서 시각 동기화(Time-sync) 및 공간 좌표계(ENU/ECEF) 변환 전처리 구현
**소요시간**: 약 2.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

이기종 센서 간의 시각 동기화(Time-sync) 오차를 보정하고 각 센서의 국소 좌표계를 공통 전술 좌표계(ENU/ECEF)로 변환하는 시공간 전처리 모듈을 구현한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **센서마다 다른 시계 오차를 보정하고 센서 국소 극좌표(거리·방위)를 공통 전술 좌표(ENU)로 바꾸는 시공간 전처리 모듈을 구현한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **기준 신호(heartbeat)와 센서 시각 차이의 중앙값으로 센서별 시계 오차를 추정 수행하고 검증하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/preprocess/ops02_preprocessor.py 및 전처리 전후 데이터셋` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 전술 환경 특유의 노이즈(스푸핑, 반사파, 센서 단절 등) 제거 타당성

#### 미션 PASS 조건

- 정량 기준: 데이터 정제 처리 속도 1,000건/초 이상, 전처리 오류율 0%


- 진행 기준: 정제 파이프라인 자동화 및 단위 테스트 케이스 5종 통과


- 검증 방법: pytest 기반 전처리 정합성 테스트 및 결측치 보정율 확인


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""이종 센서 경보의 정제(중복·결측·반사파 제거), 시각 동기화(Time-sync) 오차 보정, 센서 국소 극좌표 -> 공통 ENU 좌표 변환(가상 기준점의 위경도·ECEF 변환 포함)을 구현하고 처리 속도·정확도를 단위 시험으로 확인한다."""
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------------
# [교육용가상자료] 다중 센서(레이더·EO/IR·음향) 경보 생성기 (실제 장비 제원·배치와 무관)
# 가상 방어 지역(원점 기준 동/북 km)에 비행체 궤적을 만들고, 센서마다 '자기 위치 기준
# 거리·방위'와 '자기 시계 기준 시각'으로 경보를 낸다. 같은 비행체가 여러 센서에 동시에
# 잡히므로 경보가 중복되고, 센서마다 시계 오차·측정 오차·오경보가 다르다.
# ---------------------------------------------------------------------------
SENSORS = {
    "RDR-A": {"type": "radar", "pos": (0.0, 0.0), "sr": 0.08, "saz": 0.4, "pd": 0.95, "fa": 1.0, "max_r": 40.0, "clock": 0.0},
    "RDR-B": {"type": "radar", "pos": (16.0, -6.0), "sr": 0.10, "saz": 0.5, "pd": 0.93, "fa": 1.0, "max_r": 40.0, "clock": 0.42},
    "EOIR-C": {"type": "eoir", "pos": (5.0, 9.0), "sr": 0.40, "saz": 0.15, "pd": 0.88, "fa": 0.5, "max_r": 18.0, "clock": -0.65},
    "ACU-D": {"type": "acoustic", "pos": (10.0, 2.0), "sr": 0.60, "saz": 2.0, "pd": 0.80, "fa": 0.6, "max_r": 9.0, "clock": 0.93},
}
CENTER = np.array([8.0, 2.0])                         # 가상 방어 지역 중심
KINDS = {"uav": (0.03, 0.06), "heli": (0.04, 0.07), "fast": (0.15, 0.20)}   # 속도 범위(km/s, 추상값)
WEATHER = {"clear": {"eoir": 1.0, "acoustic": 1.0, "radar": 1.0, "fa": 1.0},
           "night": {"eoir": 0.9, "acoustic": 1.0, "radar": 1.0, "fa": 1.1},
           "fog": {"eoir": 0.55, "acoustic": 1.0, "radar": 1.0, "fa": 1.2},
           "rain": {"eoir": 0.7, "acoustic": 0.6, "radar": 0.97, "fa": 1.6}}


def _targets(rng, n, T, special):
    out, kinds = {}, {}
    for i in range(n):
        kind = "fast" if special == "fast" and i < 2 else str(rng.choice(["uav", "uav", "heli", "fast"]))
        ang = rng.uniform(0, 2 * np.pi)
        p = CENTER + rng.uniform(6, 12) * np.array([np.sin(ang), np.cos(ang)])
        hd = np.arctan2(*(CENTER - p)) + rng.normal(0, 0.35)          # 방어 지역 쪽으로(북 기준 방위)
        sp = rng.uniform(*KINDS[kind])
        omega = rng.uniform(0.04, 0.08) * rng.choice([-1, 1]) if (special == "turn" and i < 3) or rng.random() < 0.25 else 0.0
        if special == "formation" and i == 0:
            omega = 0.0
        if special == "cross" and i < 2:                              # 두 표적이 중심에서 교차
            p = CENTER + np.array([-6.0, 0.0]) if i == 0 else CENTER + np.array([0.0, -6.0])
            hd, sp, omega, kind = (np.pi / 2 if i == 0 else 0.0), 0.1, 0.0, "heli"
        if special == "formation" and i == 1:                         # 0.4 km 간격 편대
            p0 = out["TGT00"][0]
            p, hd, sp, omega, kind = p0[:2] + np.array([0.4, 0.0]), np.arctan2(p0[2], p0[3]), np.hypot(p0[2], p0[3]), 0.0, kinds["TGT00"]
        traj = np.zeros((T, 4))
        for t in range(T):
            if t > 0:
                if 20 <= t < 40:
                    hd += omega                                       # 20~40초 선회 기동
                p = p + sp * np.array([np.sin(hd), np.cos(hd)])
            traj[t] = [p[0], p[1], sp * np.sin(hd), sp * np.cos(hd)]
        out[f"TGT{i:02d}"], kinds[f"TGT{i:02d}"] = traj, kind
    return out, kinds


def make_scenario(seed, n=6, T=60, weather="clear", fa_mult=1.0, dropout=None, ghost=False, special=None,
                  noise_mult=None, drift=0.0):
    """시나리오 1개: 정답 궤적, 센서 경보(보고) 목록, 시계 동기화용 기준 신호(heartbeat)를 만든다.
    dropout=(센서, 시작, 끝): 통신 단절, ghost: 한 센서에만 보이는 거짓 항적(기만·스푸핑),
    noise_mult={센서: 배수}: 재밍으로 측정 잡음 증가, drift: 음향 센서 시계가 초당 drift초씩 흐트러짐."""
    rng = np.random.default_rng(seed)
    targets, kinds = _targets(rng, n, T, special)
    w = WEATHER[weather]
    reports, hb, mid = [], [], 0
    gpos = CENTER + np.array([-9.0, 7.0])
    for t in range(T):
        for sid, s in SENSORS.items():
            clock = s["clock"] + (drift * t if sid == "ACU-D" else 0.0)
            if t % 10 == 0:
                hb.append({"sensor": sid, "t_ref": float(t), "t_sensor": t + clock + rng.normal(0, 0.02)})
            if dropout and sid == dropout[0] and dropout[1] <= t < dropout[2]:
                continue
            nm = (noise_mult or {}).get(sid, 1.0)
            objs = [(tid, tr[t, :2]) for tid, tr in targets.items()]
            if ghost and sid == "EOIR-C":
                objs.append(("GHOST", gpos + 0.12 * t * np.array([0.6, -0.8])))
            for tid, xy in objs:
                d = xy - np.array(s["pos"])
                r = float(np.hypot(*d))
                if r > s["max_r"] or rng.random() > s["pd"] * w[s["type"]]:
                    continue
                reports.append({"msg_id": f"M{mid:05d}", "sensor": sid, "stype": s["type"],
                                "t_sensor": round(t + clock + rng.normal(0, 0.02), 3),
                                "range_km": round(r + rng.normal(0, s["sr"] * nm), 3),
                                "az_deg": round(float((np.degrees(np.arctan2(d[0], d[1])) + rng.normal(0, s["saz"] * nm)) % 360), 3),
                                "sr": s["sr"] * nm, "saz": s["saz"] * nm, "truth": tid})
                mid += 1
            for _ in range(rng.poisson(s["fa"] * fa_mult * w["fa"] * (3 if nm > 1 else 1))):   # 오경보(클러터·반사)
                reports.append({"msg_id": f"M{mid:05d}", "sensor": sid, "stype": s["type"],
                                "t_sensor": round(t + clock + rng.normal(0, 0.02), 3),
                                "range_km": round(rng.uniform(0.5, s["max_r"]), 3), "az_deg": round(rng.uniform(0, 360), 3),
                                "sr": s["sr"] * nm, "saz": s["saz"] * nm, "truth": "FA"})
                mid += 1
    return {"targets": targets, "kinds": kinds, "reports": reports, "heartbeats": hb, "T": T, "weather": weather}
# ---------------------------------------------------------------------------

# ---- 시공간 전처리 공통 함수: 정제 -> 시각 동기화 -> 공통 좌표(ENU, km) 변환 ----
def estimate_offsets(heartbeats):
    """센서별 시계 오차 = 기준 신호(heartbeat) 대비 센서 시각 차이의 중앙값(잡음에 강함)."""
    df = pd.DataFrame(heartbeats)
    return (df["t_sensor"] - df["t_ref"]).groupby(df["sensor"]).median().to_dict()


def clean(reports):
    """중복 메시지·결측·범위 밖(반사파 등 최대 탐지거리 초과) 경보를 걸러낸다. (정제 목록, 사유별 건수)"""
    seen, out, why = set(), [], {"중복": 0, "결측": 0, "범위 밖": 0}
    for r in reports:
        if r["msg_id"] in seen:
            why["중복"] += 1; continue
        seen.add(r["msg_id"])
        if any(r.get(k) is None or (isinstance(r.get(k), float) and np.isnan(r[k])) for k in ("t_sensor", "range_km", "az_deg")):
            why["결측"] += 1; continue
        if not 0 <= r["range_km"] <= SENSORS[r["sensor"]]["max_r"] * 1.1:
            why["범위 밖"] += 1; continue
        out.append(r)
    return out, why


def to_enu(r, offsets):
    """센서 국소 극좌표(거리·방위) -> 공통 ENU(동·북 km), 센서 시각 -> 기준 시각."""
    s, a = SENSORS[r["sensor"]], np.radians(r["az_deg"])
    return {**r, "t": r["t_sensor"] - offsets.get(r["sensor"], 0.0),
            "x": s["pos"][0] + r["range_km"] * np.sin(a), "y": s["pos"][1] + r["range_km"] * np.cos(a)}


def meas_cov(r):
    """거리·방위 오차를 ENU 공분산(2x2)으로 변환한다(1차 근사)."""
    a, rr = np.radians(r["az_deg"]), r["range_km"]
    J = np.array([[np.sin(a), rr * np.cos(a)], [np.cos(a), -rr * np.sin(a)]])
    return J @ np.diag([r["sr"] ** 2, np.radians(r["saz"]) ** 2]) @ J.T


def prepare(scn, sync=True):
    reps, _ = clean(scn["reports"])
    off = estimate_offsets(scn["heartbeats"]) if sync else {}
    return sorted((to_enu(r, off) for r in reps), key=lambda r: r["t"])

# 1) 위경도(LLA) <-> ECEF <-> ENU 변환 (WGS84 타원체 공식)
A, E2 = 6378137.0, 6.69437999014e-3
ORIGIN = (10.0, 20.0, 0.0)          # [교육용가상자료] 가상 기준점 위경도(실제 위치 아님)


def lla_to_ecef(lat, lon, h):
    la, lo = np.radians(lat), np.radians(lon)
    n = A / np.sqrt(1 - E2 * np.sin(la) ** 2)
    return np.array([(n + h) * np.cos(la) * np.cos(lo), (n + h) * np.cos(la) * np.sin(lo), (n * (1 - E2) + h) * np.sin(la)])


def _rot(lat, lon):
    la, lo = np.radians(lat), np.radians(lon)
    return np.array([[-np.sin(lo), np.cos(lo), 0],
                     [-np.sin(la) * np.cos(lo), -np.sin(la) * np.sin(lo), np.cos(la)],
                     [np.cos(la) * np.cos(lo), np.cos(la) * np.sin(lo), np.sin(la)]])


def ecef_to_enu(p, origin=ORIGIN):
    return _rot(*origin[:2]) @ (p - lla_to_ecef(*origin))


def enu_to_ecef(enu, origin=ORIGIN):
    return _rot(*origin[:2]).T @ enu + lla_to_ecef(*origin)


def ecef_to_lla(p):
    lon = np.arctan2(p[1], p[0]); r = np.hypot(p[0], p[1]); lat = np.arctan2(p[2], r * (1 - E2))
    for _ in range(5):
        n = A / np.sqrt(1 - E2 * np.sin(lat) ** 2)
        h = r / np.cos(lat) - n
        lat = np.arctan2(p[2], r * (1 - E2 * n / (n + h)))
    return np.degrees(lat), np.degrees(lon), h


print("[센서 설치 지점(가상 기준점 기준, 실제 위치 아님): 가상 ENU -> 위경도 -> ECEF -> ENU 왕복 변환]")
rt = []
for sid, s in SENSORS.items():
    enu = np.array([s["pos"][0] * 1000, s["pos"][1] * 1000, 30.0])
    lla = ecef_to_lla(enu_to_ecef(enu))
    back = ecef_to_enu(lla_to_ecef(*lla))
    rt.append(np.linalg.norm(back - enu))
    print(f"{sid:7s} 위도 {lla[0]:.5f} 경도 {lla[1]:.5f} 고도 {lla[2]:6.1f} m  왕복 오차 {rt[-1] * 1000:.3f} mm")

# 2) [교육용가상자료] 원시 경보 + 전술 잡음 주입(재전송 중복, 결측, 반사파 거리값, 오차값 누락)
scn = make_scenario(seed=2, n=8, T=60, special="fast")
rng = np.random.default_rng(2)
raw = [dict(r) for r in scn["reports"]]
n0 = len(raw)
raw += [dict(raw[i]) for i in rng.choice(n0, int(0.03 * n0), replace=False)]            # 재전송 중복 3%
for i in rng.choice(n0, int(0.02 * n0), replace=False):
    raw[i]["az_deg"] = None                                                           # 측정값 결측 2%
for i in rng.choice(n0, int(0.01 * n0), replace=False):
    raw[i]["range_km"] = SENSORS[raw[i]["sensor"]]["max_r"] * rng.uniform(1.5, 3)     # 반사파(다중경로) 1%
miss_sigma = rng.choice(n0, int(0.05 * n0), replace=False)
for i in miss_sigma:
    raw[i]["sr"] = None                                                               # 오차값 누락 5% -> 등록부 값으로 보정
rng.shuffle(raw)                                                                      # 도착 순서 뒤섞임

t0 = time.perf_counter()
filled = 0
for r in raw:
    if r["sr"] is None:
        r["sr"] = SENSORS[r["sensor"]]["sr"]; filled += 1
cleaned, why = clean(raw)
off = estimate_offsets(scn["heartbeats"])
enu = sorted((to_enu(r, off) for r in cleaned), key=lambda r: r["t"])
dt = time.perf_counter() - t0
print(f"\n원시 경보 {len(raw)}건 -> 정제 {len(enu)}건, 제거 사유 {why}, 오차값 보정 {filled}건(보정률 {filled / len(miss_sigma):.0%})")
print(f"처리 속도 {len(raw) / dt:,.0f} 건/초 (목표 1,000건/초 이상: {'달성' if len(raw) / dt >= 1000 else '미달'})")

# 3) 시각 동기화: 기준 신호로 추정한 센서 시계 오차
print("\n센서    실제 시계오차(s)  추정값(s)  추정 오차(s)")
for sid, s in SENSORS.items():
    print(f"{sid:7s} {s['clock']:14.3f} {off[sid]:10.3f} {abs(off[sid] - s['clock']):11.3f}")


def along_bias(reps, use_t):
    """고속 표적 경보가 정답 위치보다 진행 방향으로 얼마나 앞서거나 뒤처지는지(m, 평균). 시계 오차 x 속도만큼 생긴다."""
    e = []
    for r in reps:
        if scn["kinds"].get(r["truth"]) == "fast":
            tr, ts = scn["targets"][r["truth"]], np.arange(scn["T"])
            g = np.array([np.interp(r[use_t], ts, tr[:, j]) for j in range(4)])
            v = g[2:] / np.linalg.norm(g[2:])
            e.append(np.dot([r["x"] - g[0], r["y"] - g[1]], v))
    return float(np.mean(e)) * 1000


print("\n[고속 표적 경보의 진행 방향 위치 편향: 시계 오차 x 속도만큼 어긋난다]")
print("센서    동기화 전(m)  동기화 후(m)")
bias = {}
for sid in SENSORS:
    rs = [r for r in enu if r["sensor"] == sid]
    bias[sid] = (along_bias(rs, "t_sensor"), along_bias(rs, "t"))
    print(f"{sid:7s} {bias[sid][0]:+11.0f} {bias[sid][1]:+12.0f}")

# 4) 단위 시험 5종 (pytest 없이 assert로 수행)
tests = {
    "ENU 왕복 변환 오차 1 mm 미만": max(rt) < 1e-3,
    "방위 0도 = 정북(+y) 방향": np.allclose([to_enu({"sensor": "RDR-A", "t_sensor": 0, "range_km": 5, "az_deg": 0}, {})[k] for k in "xy"], [0, 5]),
    "중복 메시지 전량 제거": len({r["msg_id"] for r in enu}) == len(enu),
    "반사파(최대거리 초과) 전량 제거": all(r["range_km"] <= SENSORS[r["sensor"]]["max_r"] * 1.1 for r in enu),
    "시계오차 추정 오차 0.05초 미만": all(abs(off[s] - SENSORS[s]["clock"]) < 0.05 for s in SENSORS),
    "출력 시각 오름차순 정렬": all(a["t"] <= b["t"] for a, b in zip(enu, enu[1:])),
}
for k, v in tests.items():
    print(f"[{'PASS' if v else 'FAIL'}] {k}")
print(f"단위 시험 {sum(tests.values())}/{len(tests)} 통과, 전처리 오류율 {1 - sum(tests.values()) / len(tests):.0%}")

# 5) 그림: 센서별 진행 방향 위치 편향(동기화 전/후)
sids = list(SENSORS)
x = np.arange(len(sids))
plt.figure(figsize=(7, 4))
plt.bar(x - 0.2, [bias[s][0] for s in sids], 0.4, label="before time-sync", color="tab:gray")
plt.bar(x + 0.2, [bias[s][1] for s in sids], 0.4, label="after time-sync", color="tab:blue")
plt.axhline(0, color="k", lw=0.8)
plt.xticks(x, sids); plt.ylabel("along-track bias (m)"); plt.legend()
plt.title("Fast-target along-track bias by sensor (synthetic)")
plt.tight_layout(); plt.show()


<details>
<summary><b>💡 [참고] 미션 02 예상 실행 결과 (Output)</b></summary>

```text
[센서 설치 지점(가상 기준점 기준, 실제 위치 아님): 가상 ENU -> 위경도 -> ECEF -> ENU 왕복 변환]
RDR-A   위도 10.00000 경도 20.00000 고도   30.0 m  왕복 오차 0.000 mm
RDR-B   위도 9.94572 경도 20.14591 고도   52.9 m  왕복 오차 0.000 mm
EOIR-C  위도 10.08136 경도 20.04562 고도   38.4 m  왕복 오차 0.000 mm
ACU-D   위도 10.01807 경도 20.09121 고도   38.2 m  왕복 오차 0.000 mm

원시 경보 1815건 -> 정제 1712건, 제거 사유 {'중복': 52, '결측': 35, '범위 밖': 16}, 오차값 보정 88건(보정률 100%)
처리 속도 112,246 건/초 (목표 1,000건/초 이상: 달성)

센서    실제 시계오차(s)  추정값(s)  추정 오차(s)
RDR-A            0.000     -0.009       0.009
RDR-B            0.420      0.418       0.002
EOIR-C          -0.650     -0.656       0.006
ACU-D            0.930      0.938       0.008

[고속 표적 경보의 진행 방향 위치 편향: 시계 오차 x 속도만큼 어긋난다]
센서    동기화 전(m)  동기화 후(m)
RDR-A            -6           -8
RDR-B           -71           -6
EOIR-C          +95          -10
ACU-D          -179          -33
[PASS] ENU 왕복 변환 오차 1 mm 미만
[PASS] 방위 0도 = 정북(+y) 방향
[PASS] 중복 메시지 전량 제거
[PASS] 반사파(최대거리 초과) 전량 제거
[PASS] 시계오차 추정 오차 0.05초 미만
[PASS] 출력 시각 오름차순 정렬
단위 시험 6/6 통과, 전처리 오류율 0%
```
</details>
